# 17 — Sintéticos III: laboratorio de detectores con régimen conocido

> **Estado: ESQUELETO.** Este notebook fija el diseño antes de implementar nada. Todas las secciones de trabajo son celdas `TODO` controladas por el flag `IMPLEMENTADO` (por defecto `False`): con `False` el notebook se ejecuta de principio a fin sin error y solo informa de lo pendiente; con `True` cada `TODO` sin implementar levanta `NotImplementedError`. Ninguna cifra de este texto está escrita a mano: las que aparezcan saldrán de celdas.

## 0. Objetivo

Medir el comportamiento de los detectores del benchmark sobre trayectorias con **régimen
conocido** generadas por los generadores admitidos en `16`, donde sí existe verdad-terreno
exacta (en datos reales solo hay ventanas de crisis aproximadas).

## Preguntas que responde

1. ¿Recupera cada detector la secuencia de régimen verdadera (precisión, recall por evento,
   retraso de detección, falsas alarmas)?
2. ¿Cómo degrada con la dificultad del escenario (duración de crisis, separación entre
   regímenes, colas, cambios de correlación)?
3. ¿El ranking de detección en laboratorio coincide con el ranking real ADR-003?
4. ¿Qué detectores solo funcionan cuando el generador comparte su familia (circularidad)?

## Entradas y salidas (vía `regimenes.rutas`)

| papel | ruta |
|---|---|
| trayectorias admitidas | `rutas.DATA_SINTETICOS` |
| veredicto de admisión | `rutas.RESULTS_SINTETICOS / 'validacion'` |
| especificación de detectores | `regimenes.detectores.registry` (`rutas.BENCHMARK_SPEC`) |
| ranking real de referencia (solo lectura, al final) | `rutas.RESULTS_BENCHMARK` |
| resultados del laboratorio | `rutas.RESULTS_SINTETICOS / 'laboratorio'` |

## Metodología prevista

- **Escenarios**: rejilla de factores de dificultad (duración media de crisis, distancia entre
  regímenes, grados de libertad de las colas, salto de correlación) fijada en la configuración;
  varias trayectorias por celda y semilla fija.
- **Ejecución**: mismo protocolo walk-forward causal del benchmark
  (`regimenes.evaluacion.walk_forward`) y mismas especificaciones de detector
  (`regimenes.detectores.detector_specs`), sustituyendo el panel real por la trayectoria sintética.
- **Métricas**: las del benchmark (`regimenes.evaluacion.metricas`) usando como ventanas de crisis
  los tramos de régimen verdadero; además exactitud de estado alineado y retraso medio.
- **Lectura**: matriz generador × detector con intervalo por *bootstrap* entre trayectorias;
  la diagonal "misma familia" se reporta aparte; correlación de rangos con el ranking real ADR-003.

## Riesgos

- **Fuga temporal.** Todo generador se ajusta **solo** con el tramo de entrenamiento
  (`entrenamiento.fin_train` de `configs/sinteticos.yaml`); normalizaciones, selección de
  hiperparámetros y etiquetas de régimen de referencia también se calculan solo con ese tramo.
  Mientras `fin_train` sea `null` no se puede pasar `IMPLEMENTADO=True` (la celda de
  configuración lo comprueba).
- **Circularidad.** Si las etiquetas de régimen con las que se condiciona el generador salen de un
  detector del benchmark, ese detector (y su familia) juega en casa al evaluarse sobre lo
  generado. Se mitiga con etiquetas de referencia independientes de los detectores (ventanas de
  crisis y suelos de drawdown de `configs/benchmark_spec.yaml`, o varias etiquetas de familias
  distintas) y leyendo la matriz generador × detector con la diagonal "misma familia" aparte.
- **Optimismo de laboratorio.** Los sintéticos son más estacionarios que el real: el laboratorio
  ordena detectores por robustez, no sustituye la evaluación real OOS.
- **Coste.** Detectores lentos (D11, D12, D09) × escenarios × trayectorias: se limitará la rejilla
  y se usará la CLI paralela del benchmark como referencia de coste.

**Índice**

0. Objetivo, preguntas, entradas/salidas, metodología y riesgos
1. Configuración
2. Diseño de escenarios
3. Ejecución de detectores sobre trayectorias
4. Métricas contra la verdad-terreno
5. Matriz generador × detector y lectura por familias
6. Conclusiones


## 1. Configuración

In [1]:
# Configuración común del esqueleto (no lee results/benchmark ni ajusta nada).
import yaml
import pandas as pd
from IPython.display import display

from regimenes import rutas
from regimenes import sinteticos
from regimenes.sinteticos import Generador
from regimenes.sinteticos import registry as registro_sinteticos

IMPLEMENTADO = False  # pasar a True solo cuando las secciones TODO estén implementadas

CONFIG_SINTETICOS = yaml.safe_load(rutas.SINTETICOS_CONFIG.read_text(encoding='utf-8'))
SEMILLA = CONFIG_SINTETICOS.get('semilla')


def saltar(seccion: str, detalle: str) -> None:
    """Punto TODO controlado: informa y sigue (solo se llama con IMPLEMENTADO=False)."""
    print(f'[TODO pendiente] {seccion}: {detalle}')


def tabla_rutas(rutas_nb: dict) -> pd.DataFrame:
    """Entradas/salidas del notebook (siempre vía regimenes.rutas) y si existen ya en disco."""
    filas = [
        {'papel': papel, 'ruta': ruta.relative_to(rutas.ROOT).as_posix(), 'existe': ruta.exists()}
        for papel, ruta in rutas_nb.items()
    ]
    return pd.DataFrame(filas).set_index('papel')
from regimenes.detectores import detector_specs

RUTAS_NB = {
    'entrada: trayectorias admitidas': rutas.DATA_SINTETICOS,
    'entrada: veredicto de admisión': rutas.RESULTS_SINTETICOS / 'validacion',
    'entrada: banco de pruebas': rutas.BENCHMARK_SPEC,
    'referencia: ranking real (solo lectura)': rutas.RESULTS_BENCHMARK,
    'salida: laboratorio': rutas.RESULTS_SINTETICOS / 'laboratorio',
}
display(tabla_rutas(RUTAS_NB))
DETECTORES = {track: [spec.detector_id for spec in detector_specs(track)] for track in ('A', 'B')}
for track, ids in DETECTORES.items():
    print(f'Pista {track}: {len(ids)} detectores candidatos al laboratorio ->', ', '.join(ids))
print('Generadores registrados:', sorted(registro_sinteticos.GENERADORES) or 'ninguno todavía')


,ruta,existe
papel,,
entrada: trayectorias admitidas,data/sinteticos,True
entrada: veredicto de admisión,results/sinteticos/validacion,False
entrada: banco de pruebas,configs/benchmark_spec.yaml,True
referencia: ranking real (solo lectura),results/benchmark,True
salida: laboratorio,results/sinteticos/laboratorio,False


Pista A: 12 detectores candidatos al laboratorio -> D01, D02, D03, D04, D05, D06, D07, D08, D09, D10, D11, D12
Pista B: 12 detectores candidatos al laboratorio -> D01, D02, D03, D04, D05, D06, D07, D08, D09, D10, D11, D12
Generadores registrados: ninguno todavía


In [2]:
fin_train = CONFIG_SINTETICOS['entrenamiento'].get('fin_train')
if fin_train is None:
    mensaje = ('configs/sinteticos.yaml: entrenamiento.fin_train = null. Fijar el corte de '
               'entrenamiento antes de ajustar cualquier generador (regla anti-fuga).')
    if IMPLEMENTADO:
        raise ValueError(mensaje)
    print('[aviso]', mensaje)
else:
    print('Corte de entrenamiento de los generadores:', fin_train)


[aviso] configs/sinteticos.yaml: entrenamiento.fin_train = null. Fijar el corte de entrenamiento antes de ajustar cualquier generador (regla anti-fuga).


## 2. Diseño de escenarios

In [3]:
# TODO 17.2 — rejilla de escenarios
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. factores de dificultad y niveles desde configs/sinteticos.yaml (sección laboratorio, por añadir)
    # 2. secuencias de régimen impuestas por escenario; n trayectorias por celda con SEMILLA
    raise NotImplementedError('TODO 17.2: definir la rejilla de escenarios')
else:
    saltar('17.2', 'rejilla de escenarios')


[TODO pendiente] 17.2: rejilla de escenarios


## 3. Ejecución de detectores sobre trayectorias

In [4]:
# TODO 17.3 — walk-forward sobre sintético
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name
from regimenes import evaluacion as ev

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. for spec in detector_specs(pista): walk_forward(spec.factory(), trayectoria[spec.features], ...)
    # 2. cachear salidas en rutas.RESULTS_SINTETICOS / 'laboratorio' / 'paneles'
    raise NotImplementedError('TODO 17.3: ejecutar detectores sobre trayectorias sintéticas')
else:
    saltar('17.3', 'walk-forward sobre sintético')


[TODO pendiente] 17.3: walk-forward sobre sintético


## 4. Métricas contra la verdad-terreno

In [5]:
# TODO 17.4 — métricas con verdad-terreno
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. ventanas de crisis = tramos del régimen verdadero; métricas de regimenes.evaluacion.metricas + retraso
    raise NotImplementedError('TODO 17.4: calcular métricas contra la verdad-terreno')
else:
    saltar('17.4', 'métricas con verdad-terreno')


[TODO pendiente] 17.4: métricas con verdad-terreno


## 5. Matriz generador × detector y lectura por familias

In [6]:
# TODO 17.5 — matriz generador × detector
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. matriz con IC bootstrap entre trayectorias; diagonal misma familia aparte
    # 2. correlación de rangos con el ranking real ADR-003 (lectura de rutas.RESULTS_BENCHMARK al final)
    raise NotImplementedError('TODO 17.5: construir la matriz generador × detector')
else:
    saltar('17.5', 'matriz generador × detector')


[TODO pendiente] 17.5: matriz generador × detector


## 6. Conclusiones

Pendiente de implementación (cifras siempre impresas por celda). Siguiente: `18_sinteticos_aumento`.